# Verification of Stanza's Source Data's Accuracy

Stanza uses Universal Dependencies as a source of high quality, annotated datasets. However, the Latin datasets have been converted from a variety of standards. The PROIEL and Perseus treebanks both contain Caesar, so the training data provides Stanza with a couple different standards for the same text. 

My goal here is to evaluate, as best as possible, what the most common errors are likely to be in the training dataset. This will allow us to focus our data-cleaning processes on the most problematic cases in the dataset for the *Corpus Caesarianum* project.

The original Universal Dependencies data is available on GitHub. To clone both repositories (to whatever directory contains this repository), navigate there through your shell and enter these commands:

```bash
gh repo clone UniversalDependencies/UD_Latin-Perseus
gh repo clone UniversalDependencies/UD_Latin-PROIEL
```

These repositories contain not only the source data, but a document called **stats.xml** which has helpful statistics for each dataset. When more detail is required, we can use UD's official tools for comparing the datasets. In the same directory, call this command to clone the UD Tools repository:

```bash
gh repo clone UniversalDependencies/tools
```

Installing the Universal dependencies toolset requires an installation of Perl. Perl comes installed on most Linux and MacOS distributions, but Windows users need to [install Strawberry Perl](https://www.perl.org/get.html#win32). Once installed, use the `cpan` command to install the necessary dependencies.

```bash
cpan YAML JSON::Parse Lingua::Interset
#or, if cpanm is installed, install Lingua::Interset with that command instead
cpanm Lingua::Interset
```

The `lxml` module is required for dealing with XML files, and the `pyconll` module for the CONLLU treebank files. I install them on Python 3.14.5 here.

Installation commands:

```bash
pip install lxml pyconll
```

## Paths and Setup

We are dealing with all the CONLLU files in the repository, as the CONLLX format isn't available. Here is a full list of the files in each repository:

In [3]:
from pprint import pp
import os

paths_to_conllu = []
xml_paths = []

print(f"{'extension':>10} {'filename':>35} {'path':>50} {'is_file':>4} ")
print("-" * 94)
#note: casting DirEntry.path to string isn't necessary since the argument in os.scandir is a string, but it isn't guaranteed
for i in os.scandir("../UD_Latin-PROIEL"):
    if i.path.find(".conllu") > -1:
        paths_to_conllu.append(i.path) 
    if i.path.find(".xml") > -1:
        xml_paths.append(i.path)
    print(f"{i.name[i.name.rfind('.'):]:>10} {i.name:>35} {str(i.path):>50} {'yes' if i.is_file() else 'no':>4} ")
for i in os.scandir("../UD_Latin-Perseus"):
    if i.path.find(".conllu") > -1:
        paths_to_conllu.append(i.path) 
    if i.path.find(".xml") > -1:
        xml_paths.append(i.path)
    print(f"{i.name[i.name.rfind('.'):]:>10} {i.name:>35} {str(i.path):>50} {'yes' if i.is_file() else 'no':>4} ")

 extension                            filename                                               path is_file 
----------------------------------------------------------------------------------------------
      .git                                .git                            ../UD_Latin-PROIEL/.git   no 
.gitignore                          .gitignore                      ../UD_Latin-PROIEL/.gitignore  yes 
       .md                     CONTRIBUTING.md                 ../UD_Latin-PROIEL/CONTRIBUTING.md  yes 
      .txt                         LICENSE.txt                     ../UD_Latin-PROIEL/LICENSE.txt  yes 
       .md                           README.md                       ../UD_Latin-PROIEL/README.md  yes 
      .log                            eval.log                        ../UD_Latin-PROIEL/eval.log  yes 
   .conllu             la_proiel-ud-dev.conllu         ../UD_Latin-PROIEL/la_proiel-ud-dev.conllu  yes 
   .conllu            la_proiel-ud-test.conllu        ../UD_Latin-PROI

We saved all the CONLLU files to a list, `paths_to_conllu`, so we can work with them later.

In [4]:
print("CONLLU:")
pp(paths_to_conllu)
print("XML:")
pp(xml_paths)

CONLLU:
['../UD_Latin-PROIEL/la_proiel-ud-dev.conllu',
 '../UD_Latin-PROIEL/la_proiel-ud-test.conllu',
 '../UD_Latin-PROIEL/la_proiel-ud-train.conllu',
 '../UD_Latin-Perseus/la_perseus-ud-test.conllu',
 '../UD_Latin-Perseus/la_perseus-ud-train.conllu']
XML:
['../UD_Latin-PROIEL/stats.xml', '../UD_Latin-Perseus/stats.xml']


## Structure and Statistics of the Two Corpora

The structure of the *stats.xml* file is the same in each case, with a series of top-level elements for each type of tag, and sub-elements in each one showing the frequency of different values for that tag.

Here is an example structure:

```xml
<treebank>
<!-- Statistics of universal POS tags. The comments show the most frequent lemmas. -->
  <tags unique="14">
    <tag name="ADJ">11475</tag><!-- magnus, multus, bonus, publicus, sanctus, totus, primus, reliquus, alter, ceterus -->
    <tag name="ADP">16020</tag><!-- in, ad, ab, de, ex, cum, per, propter, super, pro -->
      <!-- ...-->
    <tag name="X">103</tag><!-- greek.expression -->
  </tags>
  <!-- Statistics of features and values. The comments show the most frequent word forms. -->
  <feats unique="44">
    <feat name="Aspect" value="Imp" upos="AUX,VERB">5023</feat><!-- erat, esset, erant, dicebant, essent, dicebat, posset, possent, habebat, habebant -->
    <feat name="Aspect" value="Perf" upos="AUX,VERB">13462</feat><!-- dixit, fuit, venit, factum, dixerunt, misit, facta, fecit, respondit, dedit -->
    <!-- ... -->
    <feat name="Voice" value="Pass" upos="AUX,VERB">9766</feat><!-- factum, fieri, facta, scriptum, factus, locutus, videtur, data, loqui, profectus -->
  </feats>
  <!-- Statistics of universal dependency relations. -->
  <deps unique="37">
    <dep name="acl">5373</dep>
    <!-- ... -->
    <dep name="xcomp">3467</dep>
  </deps>
</treebank>
```


Let's start by printing a side-by-side comparison of each. To facilitate this, let's create a function for turning this XML into a data frame. The data doesn't lend itself to automatic loading with pd.read_xml(), but we can still store them in arrays. I'll do this by importing `pandas` and `numpy`, which can intuitively can be installed by the same name:

```bash
python -m pip install numpy pandas
```

In [5]:
from lxml import etree
import pandas as pd
import numpy as np

def tostring_splitlines(root: Element, tag: str) -> list[str]:
    """
    Accepts a root element of stats.xml and a tag as arguments, and returns a list of strings including that element and all its subelements. 
    Each string in the list is a single line.
    """
    return etree.tostring(root.find(tag), pretty_print = True).decode().splitlines()

# TODO Refactor the code below to create a dataframe from the XML files, so it can be a bit easier to deal with. 

tag = "tags"

def tag_stats_to_df(tag):
    with open(xml_paths[0], 'r') as xml_1:
        with open(xml_paths[1], 'r') as xml_2:
            proiel = etree.parse(xml_1).getroot()
            perseus = etree.parse(xml_2).getroot()
    
            pers_tags = perseus.find(tag).findall("./*")
            proiel_tags = proiel.find(tag).findall("./*")
    
            # Example
            nums = [x.text for x in pers_tags] + [x.text for x in proiel_tags]
            names = [x.get("name") for x in pers_tags] + [x.get("name") for x in proiel_tags]
            treebank = np.concatenate((np.tile("perseus", len([x for x in pers_tags])), np.tile("proiel", len([x for x in proiel_tags]))))

            # If a comment comes after an element, we need to store it next to that element
            # A solution would be to iterate over the non-comment elements, and return nothing if it's followed by a comment and a comment if so.

            # Additionally, we need to show extra rows when the series of elements has multiple attributes.
    
            return pd.DataFrame({
                "numbers":nums,
                "labs":names,
                "treebank":treebank,
            })

tag_stats_to_df(tag)
        

   numbers   labs treebank
0     2129    ADJ  perseus
1     1349    ADP  perseus
2     1851    ADV  perseus
3      369    AUX  perseus
4     1578  CCONJ  perseus
5     1724    DET  perseus
6       34   INTJ  perseus
7     6251   NOUN  perseus
8      169    NUM  perseus
9      477   PART  perseus
10    1480   PRON  perseus
11     737  PROPN  perseus
12    4507  PUNCT  perseus
13     776  SCONJ  perseus
14    5791   VERB  perseus
15       1      X  perseus
16   11475    ADJ   proiel
17   16020    ADP   proiel
18   21952    ADV   proiel
19    8101    AUX   proiel
20   15030  CCONJ   proiel
21    7701    DET   proiel
22     547   INTJ   proiel
23   41501   NOUN   proiel
24    1712    NUM   proiel
25   25411   PRON   proiel
26    7273  PROPN   proiel
27    6914  SCONJ   proiel
28   41826   VERB   proiel
29     103      X   proiel


Let's take a look at tags first. 

In [9]:
pd.options.display.max_rows = None
tag_stats_to_df("feats")

,numbers,labs,treebank
0,17,AdpType,perseus
1,56,AdvType,perseus
2,231,AdvType,perseus
3,3760,Aspect,perseus
4,2282,Aspect,perseus
5,118,Aspect,perseus
6,3087,Case,perseus
7,4717,Case,perseus
8,816,Case,perseus
9,1267,Case,perseus


It's odd that particles (represented as PART in Perseus) don't exist in PROIEL, which only uses ADV for both. In the postagged data, *non* is always parsed as a PART, not an ADV, following Perseus practice, and same for *enim*.

PROIEL removes all punctuation beforehand, so the lack of a PUNCT tag is easy to understand.

Next, the dependency relations.

In [73]:
print_tags_comparison('deps')

deps: Perseus                                                                                                                            PROIEL                                                                                                                            
------------------------------------------------------------------------------------------------------------------------------------------------------
<deps unique="48">                                                                                                                 <deps unique="37">                                                                                                                
    <dep name="acl">86</dep>                                                                                                           <dep name="acl">5373</dep>                                                                                                    
    <dep name="acl:relcl">288</dep>                      

Now, finally, the UPOS features.


In [78]:
print_tags_comparison('feats')

feats: Perseus                                                                                                                                                                              PROIEL                                                                                                                                                                              
------------------------------------------------------------------------------------------------------------------------------------------------------
<feats unique="61">                                                                                                                                                                  <feats unique="44">                                                                                                                                                                 
    <feat name="AdpType" value="Post" upos="ADP">17</feat><!-- cum -->                                                

This last one bears more exploration. Perseus has numerous tags that are covered more concisely in the PROIEL dataset.


